In [ ]:
%%spark

# ============================================================
# 7_PUBLICAR_HIVE
# ============================================================


def publicar_hive(
    df_rcm_fnc_cli_final,
    df_rcm_fnc_cli_evtl_append,
    df_ctl_oprl_rcm_final,
    database: str,
    path_save_hdfs: str,
    executar: bool,
    detalhar_recomendacao: bool = False,
) -> dict:
    logger_etapa = logger
    tabela_rcm = TABELAS_HIVE["hive_cliente_recomendacao_ativa"]
    tabela_evtl = TABELAS_HIVE["hive_cliente_recomendacao_historica"]
    tabela_ctl = TABELAS_HIVE["hive_controle_operacional_recomendacao"]
    caminhos = caminhos_temporarios_ctl(path_save_hdfs)

    df_rcm = projetar_contrato_hive(df_rcm_fnc_cli_final, tabela_rcm)
    df_evtl_append = projetar_contrato_hive(df_rcm_fnc_cli_evtl_append, tabela_evtl)
    df_ctl = projetar_contrato_hive(df_ctl_oprl_rcm_final, tabela_ctl)

    validar_sem_duplicidade(df_rcm, CONTRATO_HIVE[tabela_rcm]["chaves"], "PUBLICACAO_HIVE_RCM_FNC_CLI", logger_etapa=logger_etapa)
    validar_sem_duplicidade(df_evtl_append, CONTRATO_HIVE[tabela_evtl]["chaves"], "PUBLICACAO_HIVE_RCM_FNC_CLI_EVTL_APPEND", logger_etapa=logger_etapa)
    validar_estrutura_ctl(df_ctl, "PUBLICACAO_HIVE_CTL_OPRL_RCM", logger_etapa=logger_etapa)

    stats = {
        "dry_run": not executar,
        "qtd_rcm_fnc_cli": contar_dataframe_sql(df_rcm, "PUBLICACAO_HIVE_RCM"),
        "qtd_evtl_append": contar_dataframe_sql(df_evtl_append, "PUBLICACAO_HIVE_EVTL_APPEND"),
        "ctl_publicado": False,
    }

    detalhes_publicacao = []
    if detalhar_recomendacao:
        view_rcm = registrar_view_sql(df_rcm, "publicacao_hive_rcm_detalhe")
        view_evtl = registrar_view_sql(df_evtl_append, "publicacao_hive_evtl_detalhe")
        detalhes_publicacao = [
            row.asDict(recursive=True)
            for row in coletar_sql(f"""
                WITH ativa AS (
                    SELECT
                        {coluna_sql(COL_CD_IDFR_AVS)},
                        {coluna_sql(COL_NR_IDFR_PROJ)},
                        {coluna_sql(COL_NR_IDFR_RCM)},
                        COUNT(1) AS QTD_ATIVA
                    FROM {view_rcm}
                    GROUP BY {coluna_sql(COL_CD_IDFR_AVS)}, {coluna_sql(COL_NR_IDFR_PROJ)}, {coluna_sql(COL_NR_IDFR_RCM)}
                ), historico AS (
                    SELECT
                        {coluna_sql(COL_CD_IDFR_AVS)},
                        {coluna_sql(COL_NR_IDFR_PROJ)},
                        {coluna_sql(COL_NR_IDFR_RCM)},
                        COUNT(1) AS QTD_HISTORICO_APPEND
                    FROM {view_evtl}
                    GROUP BY {coluna_sql(COL_CD_IDFR_AVS)}, {coluna_sql(COL_NR_IDFR_PROJ)}, {coluna_sql(COL_NR_IDFR_RCM)}
                )
                SELECT
                    COALESCE(a.{coluna_sql(COL_CD_IDFR_AVS)}, h.{coluna_sql(COL_CD_IDFR_AVS)}) AS {coluna_sql(COL_CD_IDFR_AVS)},
                    COALESCE(a.{coluna_sql(COL_NR_IDFR_PROJ)}, h.{coluna_sql(COL_NR_IDFR_PROJ)}) AS {coluna_sql(COL_NR_IDFR_PROJ)},
                    COALESCE(a.{coluna_sql(COL_NR_IDFR_RCM)}, h.{coluna_sql(COL_NR_IDFR_RCM)}) AS {coluna_sql(COL_NR_IDFR_RCM)},
                    COALESCE(a.QTD_ATIVA, 0) AS QTD_ATIVA,
                    COALESCE(h.QTD_HISTORICO_APPEND, 0) AS QTD_HISTORICO_APPEND
                FROM ativa a
                FULL OUTER JOIN historico h
                    ON a.{coluna_sql(COL_CD_IDFR_AVS)} <=> h.{coluna_sql(COL_CD_IDFR_AVS)}
                ORDER BY {coluna_sql(COL_CD_IDFR_AVS)}
            """)
        ]

    for tabela in ORDEM_PUBLICACAO_HIVE:
        if not tabela_hive_existe(database, tabela):
            raise ErroAcessoDados(
                codigo="HIVE_DESTINO_INEXISTENTE",
                mensagem="Tabela Hive de destino inexistente.",
                etapa="PUBLICACAO_HIVE",
                objeto=nome_tabela_hive(database, tabela),
                acao="Criar ou disponibilizar a tabela Hive conforme o DDL esperado antes da publicacao.",
            )

        df_destino = ler_tabela_hive(database, tabela)
        schema_esperado = schema_contrato_hive(tabela)
        campos_destino = [
            (campo.name, campo.dataType.simpleString())
            for campo in df_destino.schema.fields
        ]
        campos_esperados = [
            (campo.name, campo.dataType.simpleString())
            for campo in schema_esperado.fields
        ]

        if campos_destino != campos_esperados:
            raise ErroContratoDados(
                codigo="HIVE_DESTINO_SCHEMA_INCOMPATIVEL",
                mensagem="Tabela Hive de destino possui schema incompativel.",
                etapa="PUBLICACAO_HIVE",
                objeto=nome_tabela_hive(database, tabela),
                detalhes={
                    "schema_esperado": campos_esperados,
                    "schema_encontrado": campos_destino,
                },
                acao="Alinhar o DDL da tabela Hive ao contrato antes da publicacao.",
            )

    if not executar:
        logger_etapa.info(
            "[PUBLICACAO_HIVE][DRY_RUN] Publicacao Hive simulada. "
            "Nenhuma escrita em Hive ou HDFS sera executada.",
        )
        if detalhar_recomendacao and detalhes_publicacao:
            linhas_detalhe = [
                "[DETALHE_AGRUPADO]",
                "",
                f"[AÇÃO] DRY_RUN [PUBLICACAO_HIVE] SIMULADA [QTD] {len(detalhes_publicacao)}",
                "",
            ]

            for detalhe in detalhes_publicacao:
                linhas_detalhe.append(f"    - AVISO={detalhe.get(COL_CD_IDFR_AVS)}")
                linhas_detalhe.append(
                    f"        CLIENTES     → ATIVA={detalhe.get('QTD_ATIVA')} | HISTORICO_APPEND={detalhe.get('QTD_HISTORICO_APPEND')}"
                )
                linhas_detalhe.append("        HIVE         → publicacao simulada; nenhuma escrita executada")
                linhas_detalhe.append("")

            logger_etapa.info("\n".join(linhas_detalhe))

        return {
            "hive_publicado": False,
            "stats": stats,
        }

    logger_etapa.info(
        "[PUBLICACAO_HIVE][ORDEM_CONFIRMADA] Publicacao seguira ordem segura: "
        "RCM_FNC_CLI overwrite, RCM_FNC_CLI_EVTL append idempotente, CTL_OPRL_RCM overwrite final via temp_lineage.",
    )

    lineage_removido = remover_path_hdfs_se_existir(caminhos["temp_lineage"], executar=True, logger_etapa=logger_etapa, contexto="PUBLICACAO_HIVE")

    if not lineage_removido:
        raise ErroOperacional(
            codigo="HDFS_LIMPEZA_TEMPORARIOS_FALHOU",
            mensagem="Limpeza do temp_lineage anterior nao foi concluida.",
            etapa="PUBLICACAO_HIVE",
            objeto=caminhos["temp_lineage"],
            acao="Verificar o path temporario HDFS antes de reiniciar a publicacao Hive.",
        )

    qtd_ctl = contar_dataframe_sql(df_ctl, "PUBLICACAO_HIVE_CTL")
    # EXCECAO_SQL: escrita Parquet temporaria para quebrar lineage antes do overwrite final.
    try:
        df_ctl.write.mode("overwrite").parquet(caminhos["temp_lineage"])
    except ErroPipeline:
        raise
    except Exception as exc:
        raise ErroAcessoDados(
            codigo="HDFS_ESCRITA_TEMPORARIO_FALHOU",
            mensagem="Falha ao gravar temp_lineage da tabela de controle.",
            etapa="PUBLICACAO_HIVE",
            objeto=caminhos["temp_lineage"],
            detalhes={"operacao": "OVERWRITE_PARQUET"},
            acao="Verificar permissao, espaco e disponibilidade do path HDFS temporario.",
        ) from exc
    df_ctl_lineage = ler_parquet_sql(caminhos["temp_lineage"])
    validar_estrutura_ctl(df_ctl_lineage, "PUBLICACAO_HIVE_TEMP_LINEAGE_READBACK", logger_etapa=logger_etapa)

    qtd_readback = contar_dataframe_sql(df_ctl_lineage, "PUBLICACAO_HIVE_CTL_READBACK")
    if qtd_ctl != qtd_readback:
        view_ctl_final = registrar_view_sql(df_ctl, "publicacao_hive_ctl_final")
        view_ctl_readback = registrar_view_sql(df_ctl_lineage, "publicacao_hive_ctl_readback")
        df_lineage_divergente = spark_sql(f"""
            SELECT
                COALESCE(final.{coluna_sql(COL_NR_IDFR_PROJ)}, readback.{coluna_sql(COL_NR_IDFR_PROJ)}) AS {coluna_sql(COL_NR_IDFR_PROJ)},
                COALESCE(final.{coluna_sql(COL_NR_IDFR_RCM)}, readback.{coluna_sql(COL_NR_IDFR_RCM)}) AS {coluna_sql(COL_NR_IDFR_RCM)},
                COALESCE(final.{coluna_sql(COL_NR_VRS_VLDD_RCM)}, readback.{coluna_sql(COL_NR_VRS_VLDD_RCM)}) AS {coluna_sql(COL_NR_VRS_VLDD_RCM)},
                {literal_sql('CONTAGEM_TEMP_LINEAGE')} AS campo,
                CASE
                    WHEN readback.{coluna_sql(COL_NR_IDFR_PROJ)} IS NULL THEN {literal_sql('AUSENTE_NO_READBACK')}
                    WHEN final.{coluna_sql(COL_NR_IDFR_PROJ)} IS NULL THEN {literal_sql('EXTRA_NO_READBACK')}
                    ELSE {literal_sql('DIVERGENTE')}
                END AS valor_encontrado,
                {literal_sql(f'qtd_ctl={qtd_ctl};qtd_readback={qtd_readback}')} AS limite_esperado
            FROM {view_ctl_final} final
            FULL OUTER JOIN {view_ctl_readback} readback
                ON final.{coluna_sql(COL_NR_IDFR_PROJ)} <=> readback.{coluna_sql(COL_NR_IDFR_PROJ)}
               AND final.{coluna_sql(COL_NR_IDFR_RCM)} <=> readback.{coluna_sql(COL_NR_IDFR_RCM)}
               AND final.{coluna_sql(COL_NR_VRS_VLDD_RCM)} <=> readback.{coluna_sql(COL_NR_VRS_VLDD_RCM)}
            WHERE final.{coluna_sql(COL_NR_IDFR_PROJ)} IS NULL
               OR readback.{coluna_sql(COL_NR_IDFR_PROJ)} IS NULL
        """)
        registrar_erros_recomendacao(
            df_lineage_divergente,
            logger_etapa,
            "PUBLICACAO_HIVE_TEMP_LINEAGE_READBACK",
            "CONTAGEM_DIVERGENTE",
            "temp_lineage com contagem divergente do CTL final",
            ["campo", "valor_encontrado", "limite_esperado"],
        )
        raise ErroContratoDados(
            codigo="CTL_LINEAGE_DIVERGENTE",
            mensagem="Temp_lineage possui contagem divergente da CTL final.",
            etapa="PUBLICACAO_HIVE",
            objeto=caminhos["temp_lineage"],
            detalhes={"qtd_ctl": qtd_ctl, "qtd_readback": qtd_readback},
            acao="Analisar o temporario e a CTL preparada antes de nova publicacao.",
        )

    # EXCECAO_SQL: overwrite Hive e ponto de saida fisico do pipeline.
    try:
        df_rcm.write.mode("overwrite").insertInto(nome_tabela_hive(database, tabela_rcm))
    except ErroPipeline:
        raise
    except Exception as exc:
        raise ErroAcessoDados(
            codigo="HIVE_PUBLICACAO_FALHOU",
            mensagem="Falha ao publicar tabela Hive.",
            etapa="PUBLICACAO_HIVE",
            objeto=nome_tabela_hive(database, tabela_rcm),
            detalhes={"modo": "overwrite"},
            acao="Verificar permissao, disponibilidade e schema da tabela Hive de destino.",
        ) from exc

    publicar_tabelas_ando(
        df=df_rcm,
        database=database,
        tabela=tabela_rcm,
        modo="overwrite"
    )

    if stats["qtd_evtl_append"] > 0:
        # EXCECAO_SQL: append Hive idempotente e ponto de saida fisico do pipeline.
        try:
            df_evtl_append.write.mode("append").insertInto(nome_tabela_hive(database, tabela_evtl))
        except ErroPipeline:
            raise
        except Exception as exc:
            raise ErroAcessoDados(
                codigo="HIVE_PUBLICACAO_FALHOU",
                mensagem="Falha ao publicar tabela Hive.",
                etapa="PUBLICACAO_HIVE",
                objeto=nome_tabela_hive(database, tabela_evtl),
                detalhes={"modo": "append"},
                acao="Verificar permissao, disponibilidade e schema da tabela Hive de destino.",
            ) from exc

        publicar_tabelas_ando(
            df=df_evtl_append,
            database=database,
            tabela=tabela_evtl,
            modo="append",
        )


    else:
        logger_etapa.info("[PUBLICACAO_HIVE][EVTL_VAZIO] Lote idempotente vazio. Append dispensado.")

    # EXCECAO_SQL: overwrite Hive final a partir do readback SQL do temp_lineage.
    try:
        projetar_contrato_hive(df_ctl_lineage, tabela_ctl).write.mode("overwrite").insertInto(nome_tabela_hive(database, tabela_ctl))
    except ErroPipeline:
        raise
    except Exception as exc:
        raise ErroAcessoDados(
            codigo="HIVE_PUBLICACAO_FALHOU",
            mensagem="Falha ao publicar tabela Hive.",
            etapa="PUBLICACAO_HIVE",
            objeto=nome_tabela_hive(database, tabela_ctl),
            detalhes={"modo": "overwrite"},
            acao="Verificar permissao, disponibilidade e schema da tabela Hive de destino.",
        ) from exc
    stats["ctl_publicado"] = True

    if detalhar_recomendacao and detalhes_publicacao:
        linhas_detalhe = [
            "[DETALHE_AGRUPADO]",
            "",
            f"[AÇÃO] PUBLICA_HIVE [PUBLICACAO_HIVE] EXECUTADA [QTD] {len(detalhes_publicacao)}",
            "",
        ]

        for detalhe in detalhes_publicacao:
            linhas_detalhe.append(f"    - AVISO={detalhe.get(COL_CD_IDFR_AVS)}")
            linhas_detalhe.append(
                f"        CLIENTES     → ATIVA={detalhe.get('QTD_ATIVA')} | HISTORICO_APPEND={detalhe.get('QTD_HISTORICO_APPEND')}"
            )
            linhas_detalhe.append("        HIVE         → RCM_FNC_CLI overwrite, EVTL append, CTL overwrite final")
            linhas_detalhe.append("")

        logger_etapa.info("\n".join(linhas_detalhe))

    logger_etapa.obj(stats, title="[PUBLICACAO_HIVE][FIM] Stats")

    return {
        "hive_publicado": True,
        "stats": stats,
    }